# Mammography — patient-level analysis

Patient-level reconstruction and paired statistical analysis derived from the detailed mammography prediction outputs.


In [ ]:

from pathlib import Path

METRICS_DIR = Path("/mnt/data/full_audit/ROI256_TRAINING_RESULTS/metrics")

OUTPUT_DIR = Path("/mnt/data/MAMMOGRAPHY_PATIENT_ANALYSIS_NOTEBOOK")

BOOTSTRAP_RESAMPLES = 20000
RANDOM_SEED = 20260801

print("METRICS_DIR =", METRICS_DIR)
print("OUTPUT_DIR  =", OUTPUT_DIR)
print("BOOTSTRAP_RESAMPLES =", BOOTSTRAP_RESAMPLES)
print("RANDOM_SEED =", RANDOM_SEED)


In [ ]:

import json
from itertools import combinations

import numpy as np
import pandas as pd
from scipy import stats

MODELS = ["swin_tiny_unet", "attention_unet", "unet"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK")


In [ ]:

def holm_adjust(pvalues):
    p = np.asarray(pvalues, dtype=float)
    order = np.argsort(p)
    adjusted = np.zeros_like(p)
    running = 0.0
    m = len(p)
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p[idx])
        adjusted[idx] = min(running, 1.0)
    return adjusted

def load_primary_metrics(metrics_dir: Path) -> pd.DataFrame:
    paths = sorted(metrics_dir.glob("*_detailed_metrics.csv"))
    if len(paths) != 9:
        raise RuntimeError(
            f"Expected 9 detailed-metric files, found {len(paths)} in {metrics_dir}"
        )

    frames = []
    required = {"split", "model", "seed", "patient_id", "tp", "fp", "fn", "tn"}

    for path in paths:
        df = pd.read_csv(path)
        missing = required.difference(df.columns)
        if missing:
            raise RuntimeError(f"{path.name} is missing columns: {sorted(missing)}")

        if "threshold_policy" in df.columns:
            selected = df[df["threshold_policy"] == "selected_on_cbis_validation"].copy()
            if selected.empty:
                raise RuntimeError(f"No validation-selected-threshold rows in {path.name}")
            df = selected

        df["source_file"] = path.name
        frames.append(df)

    out = pd.concat(frames, ignore_index=True)
    observed = set(out["model"].unique())
    if observed != set(MODELS):
        raise RuntimeError(f"Unexpected model set: {sorted(observed)}")

    return out

def patient_metrics(raw: pd.DataFrame):
    grouped = (
        raw.groupby(["split", "model", "seed", "patient_id"], as_index=False)[["tp", "fp", "fn", "tn"]]
        .sum()
    )

    eps = 1e-12
    grouped["dice"] = 2 * grouped["tp"] / (2 * grouped["tp"] + grouped["fp"] + grouped["fn"] + eps)
    grouped["iou"] = grouped["tp"] / (grouped["tp"] + grouped["fp"] + grouped["fn"] + eps)
    grouped["precision"] = grouped["tp"] / (grouped["tp"] + grouped["fp"] + eps)
    grouped["recall"] = grouped["tp"] / (grouped["tp"] + grouped["fn"] + eps)

    averaged = (
        grouped.groupby(["split", "model", "patient_id"], as_index=False)[["dice", "iou", "precision", "recall"]]
        .mean()
    )
    return grouped, averaged

def bootstrap_summary(averaged: pd.DataFrame, rng: np.random.Generator, n_boot: int) -> pd.DataFrame:
    rows = []
    for (split, model), group in averaged.groupby(["split", "model"]):
        x = group["dice"].to_numpy(float)
        indices = rng.integers(0, len(x), size=(n_boot, len(x)))
        boot = x[indices].mean(axis=1)
        lo, hi = np.quantile(boot, [0.025, 0.975])

        rows.append({
            "split": split,
            "model": model,
            "n_patients": int(group["patient_id"].nunique()),
            "dice_mean": float(x.mean()),
            "dice_sd_between_patients": float(x.std(ddof=1)),
            "dice_bootstrap_ci95_low": float(lo),
            "dice_bootstrap_ci95_high": float(hi),
            "iou_mean": float(group["iou"].mean()),
            "precision_mean": float(group["precision"].mean()),
            "recall_mean": float(group["recall"].mean()),
        })

    return pd.DataFrame(rows).sort_values(["split", "model"]).reset_index(drop=True)

def paired_tests(averaged: pd.DataFrame, rng: np.random.Generator, n_boot: int) -> pd.DataFrame:
    rows = []
    for split, split_df in averaged.groupby("split"):
        pivot = split_df.pivot(index="patient_id", columns="model", values="dice")
        local = []
        raw_p = []

        for model_a, model_b in combinations(MODELS, 2):
            if model_a not in pivot or model_b not in pivot:
                continue

            pair = pivot[[model_a, model_b]].dropna()
            diff = (pair[model_a] - pair[model_b]).to_numpy(float)

            boot_idx = rng.integers(0, len(diff), size=(n_boot, len(diff)))
            boot = diff[boot_idx].mean(axis=1)
            lo, hi = np.quantile(boot, [0.025, 0.975])

            _, p = stats.wilcoxon(diff, zero_method="wilcox", alternative="two-sided")

            local.append({
                "split": split,
                "model_a": model_a,
                "model_b": model_b,
                "n_patients": int(len(diff)),
                "mean_difference": float(diff.mean()),
                "median_difference": float(np.median(diff)),
                "bootstrap_ci95_low": float(lo),
                "bootstrap_ci95_high": float(hi),
                "wilcoxon_p_raw": float(p),
            })
            raw_p.append(float(p))

        adjusted = holm_adjust(raw_p)
        for row, p_holm in zip(local, adjusted):
            row["holm_p"] = float(p_holm)
            row["significant_after_holm_0_05"] = bool(p_holm < 0.05)
            rows.append(row)

    return pd.DataFrame(rows).sort_values(["split", "model_a", "model_b"]).reset_index(drop=True)

print("Fonctions chargées")


In [ ]:

if not METRICS_DIR.exists():
    raise FileNotFoundError(f"METRICS_DIR not found: {METRICS_DIR}")

files = sorted(METRICS_DIR.glob("*_detailed_metrics.csv"))
print("Nombre de fichiers détectés :", len(files))
for f in files:
    print("-", f.name)


In [ ]:

rng = np.random.default_rng(RANDOM_SEED)

raw = load_primary_metrics(METRICS_DIR)
per_seed, averaged = patient_metrics(raw)
summary = bootstrap_summary(averaged, rng, BOOTSTRAP_RESAMPLES)
comparisons = paired_tests(averaged, rng, BOOTSTRAP_RESAMPLES)

print("Analyse terminée")
print("Taille raw      :", raw.shape)
print("Taille per_seed :", per_seed.shape)
print("Taille averaged :", averaged.shape)


In [ ]:

display(raw.head())
display(per_seed.head())
display(averaged.head())


In [ ]:

display(summary)

print("\nRésumé focalisé sur CBIS test et INbreast :")
display(summary[summary["split"].isin(["test", "external_inbreast"])])


In [ ]:

display(comparisons)

print("\nComparaisons focalisées sur CBIS test et INbreast :")
display(comparisons[comparisons["split"].isin(["test", "external_inbreast"])])


In [ ]:

per_seed.to_csv(OUTPUT_DIR / "mammography_patient_metrics_by_seed.csv", index=False)
averaged.to_csv(OUTPUT_DIR / "mammography_patient_metrics_seed_averaged.csv", index=False)
summary.to_csv(OUTPUT_DIR / "mammography_patient_summary.csv", index=False)
comparisons.to_csv(OUTPUT_DIR / "mammography_patient_pairwise.csv", index=False)

metadata = {
    "analysis": "primary mammography patient-level aggregation",
    "threshold_policy": "selected_on_cbis_validation",
    "seed_aggregation": "mean of patient metrics across seeds 42, 123, 2025",
    "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
    "random_seed": RANDOM_SEED,
    "statistical_test": "two-sided Wilcoxon signed-rank",
    "multiplicity": "Holm within split across three pairwise model comparisons",
    "metrics_dir": str(METRICS_DIR),
    "output_dir": str(OUTPUT_DIR),
    "generated_files": [
        "mammography_patient_metrics_by_seed.csv",
        "mammography_patient_metrics_seed_averaged.csv",
        "mammography_patient_summary.csv",
        "mammography_patient_pairwise.csv",
    ],
}
(OUTPUT_DIR / "analysis_metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

print("Exports écrits dans :", OUTPUT_DIR)
for p in OUTPUT_DIR.iterdir():
    print("-", p.name)


## Fichiers de sortie
